# 02b - Membersihkan Data
Materi Modul 2: menangani nilai yang hilang dan pencilan (termasuk dengan residual).
Data sengaja dikotori (missing value) lalu dibersihkan, supaya tiap langkah terlihat.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

rng = np.random.default_rng(42)
df = pd.read_csv('../dataset/penguins.csv')

## 1. Simulasi missing value

In [2]:
mask = rng.random(df.shape) < 0.03
mask[:, -1] = False   # kolom target jangan dikotori
df_kotor = df.mask(mask)
df_kotor.isna().sum()

bill_length_mm       9
bill_depth_mm        7
flipper_length_mm    7
body_mass_g          5
target               0
dtype: int64

## 2. Imputasi (median)

In [3]:
imp = SimpleImputer(strategy='median')
X = pd.DataFrame(imp.fit_transform(df_kotor.drop(columns='target')), columns=df.columns[:-1])
print('Missing setelah imputasi:', X.isna().sum().sum())

Missing setelah imputasi: 0


## 3. Deteksi outlier (IQR)

In [4]:
Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
IQR = Q3 - Q1
outlier = ((X < Q1 - 1.5*IQR) | (X > Q3 + 1.5*IQR)).sum()
print('Jumlah outlier per fitur:'); print(outlier)

Jumlah outlier per fitur:
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
dtype: int64


## 4. Scaling

In [5]:
X_std = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
X_mm  = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

fig, ax = plt.subplots(1, 3, figsize=(15,4))
for a, d, t in zip(ax, [X, X_std, X_mm], ['Asli','StandardScaler','MinMaxScaler']):
    d[['body_mass_g','flipper_length_mm']].boxplot(ax=a); a.set_title(t)
plt.show()

## 3b. Pencilan berbasis residual
Fit regresi linear sederhana, lalu tandai titik dengan standardized residual > 3.

In [6]:
from sklearn.linear_model import LinearRegression

xr, yr = X[['flipper_length_mm']], X['body_mass_g']
lr = LinearRegression().fit(xr, yr)
res = yr - lr.predict(xr)
z = (res - res.mean()) / res.std()
pencilan = z.abs() > 3
print('Jumlah pencilan (|z residual| > 3):', pencilan.sum())
plt.scatter(xr, yr, c=pencilan, cmap='coolwarm')
plt.xlabel('flipper_length_mm'); plt.ylabel('body_mass_g'); plt.title('Pencilan berdasarkan residual'); plt.show()

Jumlah pencilan (|z residual| > 3): 2


In [7]:
# Opsi penanganan: hapus pencilan
X_bersih = X.loc[~pencilan].reset_index(drop=True)
print(X.shape, '->', X_bersih.shape)

(333, 4) -> (331, 4)


## Catatan
- (kenapa pilih median, kapan pakai StandardScaler vs MinMax, dll.)